[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Reshaping


## What you will be able to do

Turn the long table of readings into a wide one with a column for each station, and back again, and
choose the shape that makes the question you are asking a single line of code.

## The idea

### The problem

`readings` is long, with one row for each station at each hour, 35,040 rows in all, and a column
that says which station a reading came from. That shape suits `groupby` and filtering, and it suits
nothing that compares two stations at the same hour. "How much warmer was Bergen than Oslo at noon
on 1 June?" needs Bergen's reading and Oslo's in the same row, and in the long table they are two
rows apart.

A report has the opposite problem. A reader wants a small table with the stations down the side and
the months across the top, the way a spreadsheet lays it out, and a groupby by station and month
returns 48 rows in a single column. A spreadsheet that arrives wide, with a column for each month,
has the problem in reverse. Before `groupby` can average all the months at once, the month names
have to stop being column names and become values in a column.

By hand, each of those moves is a loop that builds a dictionary of dictionaries, and pandas does
each in one call.

### Long and wide

> A **long** table has one row for each observation and a column for each thing recorded about it,
> so the station is a value in a `station` column. A **wide** table has one row for each subject,
> such as an hour or a station, and one column for each value of a variable, so each station is a
> column of its own. **Reshaping** moves a table between the two: `pivot` and `pivot_table` make a
> long table wide, `melt` makes a wide table long, and `unstack` and `stack` move a level of the
> index into the columns and back.

### Why it works that way

`pivot` takes three columns of a long table, one for the new row labels, one for the new column
labels and one for the values, and puts each value at the crossing of its row and its column. Each
crossing can hold one value, so no pair of row label and column label can appear more than once.
Where a pair appears many times, as every station and month does in the readings, `pivot_table` is
the function to use. It collects the values that fall in each cell and combines them, with a mean
unless you name another function.

`melt` undoes `pivot`. The columns you name in `id_vars` stay as they are, and every other column
becomes a pair of values in two new columns, its name in one and its value in the other.

`unstack` and `stack` work on the index. A groupby by two keys returns a Series whose index has two
levels, and `unstack` turns the inner level into columns. `stack` turns the columns back into an
inner level of the index. In pandas 2.2 it also drops missing values unless you ask it not to, which
the Common errors section shows.

### Where this shows up

Spreadsheets and printed reports are usually wide. seaborn's plotting functions work best with long
tables, and the readings table in the **sqlite3, Deep Dive** guide is long too. A groupby with two
keys in the **groupby** and **Aggregating** notebooks gives a two-level index that `unstack` lays
out as a table, as the **Why Pandas** notebook did for the monthly means. The **Merging** notebook
joins tables in either shape on a key column.

### What this notebook covers

- the readings made wide with `pivot`, one column per station
- arithmetic between stations on the wide table
- the wide table made long again with `melt`
- a report of monthly means, and a count of frost hours, with `pivot_table`
- `unstack` and `stack` on a two-level index
- four errors, two of which change the numbers without a message

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things.

```python
import pandas as pd

long = pd.DataFrame({
    "station": ["Bergen", "Oslo", "Bergen", "Oslo"],
    "hour": ["00:00", "00:00", "01:00", "01:00"],
    "celsius": [8.2, 6.1, 7.9, 5.8],
})
wide = long.pivot(index="hour", columns="station", values="celsius")
print(wide)
print(wide.reset_index().melt(id_vars="hour", value_name="celsius"))
```

```
station  Bergen  Oslo
hour
00:00       8.2   6.1
01:00       7.9   5.8
    hour station  celsius
0  00:00  Bergen      8.2
1  01:00  Bergen      7.9
2  00:00    Oslo      6.1
3  01:00    Oslo      5.8
```

`pivot` made a row for each hour and a column for each station, so both stations' readings for an
hour now sit in the same row. `melt` turned the two station columns back into rows, and named the
new column `station` after the name `pivot` had given the columns.

## Setup

One library, three helpers for building the readings, and two tables.

- `pandas`, imported as `pd`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings

`year_of_readings` makes a reading for every hour of 2025 at the four stations in `STATIONS`, by the
same formula as the **sqlite3, Deep Dive** guide, with Svalbard's 24 readings for 2 March missing.
`readings` is the long table of all 35,040, with the columns `station`, `time` and `celsius`, and
`stations` holds each station's latitude, with Kirkenes, which has no readings.

In [1]:
import math
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


## Worked examples

### A column for each station with pivot

`time` becomes the row labels, the station names become the columns, and each reading goes where its
hour and its station cross:

In [2]:
wide = readings.pivot(index="time", columns="station", values="celsius")

print(wide.shape)
wide.head(3)

(8760, 4)


station,Bergen,Oslo,Svalbard,Tromso
time,,,,
2025-01-01 00:00:00,-3.6,-3.5,-14.6,-6.7
2025-01-01 01:00:00,-3.7,-5.3,-16.4,-8.5
2025-01-01 02:00:00,-3.7,-5.3,-16.4,-8.5


8,760 rows, one for each hour of the year, and four columns, one for each station. The label
`station` above the columns is the name of the column index, and `time` is the name of the row
index. Svalbard's missing hours are `NaN` in its column, so the wide table holds the same 35,040
cells, 24 of them empty:

In [3]:
wide.isna().sum()

station
Bergen       0
Oslo         0
Svalbard    24
Tromso       0
dtype: int64

### Arithmetic between stations

On the wide table, a comparison between two stations at every hour is a subtraction of two columns,
and pandas lines the rows up by `time`:

In [4]:
raw = wide["Bergen"] - wide["Oslo"]
print("smallest raw difference:", raw.min())

difference = raw.round(1)
print("Bergen minus Oslo, on average:", round(difference.mean(), 2))
difference.value_counts()

smallest raw difference: -0.10000000000000142
Bergen minus Oslo, on average: 1.5


 1.6    8244
-0.1     516
Name: count, dtype: int64

A difference of two decimals carries the rounding error of each, so the smallest raw difference
printed as -0.10000000000000142, and `.round(1)` turns it back into -0.1 before counting. Bergen was
1.6 degrees warmer than Oslo in 8,244 hours and 0.1 degrees colder in 516, and those are the only
two values. The formula's wobble repeats every 17 hours, so the gap between two stations takes very
few values. On the long table the same question would need a filter for each station and a join on
`time` to put the two readings side by side.

### The wide table made long again with melt

`reset_index` turns `time` back into an ordinary column, so that `melt` can keep it, and every other
column becomes rows:

In [5]:
long = wide.reset_index().melt(id_vars="time", var_name="station", value_name="celsius")

print(long.shape)
long.head(3)

(35040, 3)


,time,station,celsius
0,2025-01-01 00:00:00,Bergen,-3.6
1,2025-01-01 01:00:00,Bergen,-3.7
2,2025-01-01 02:00:00,Bergen,-3.7


35,040 rows again, with all of Bergen's year first and then Oslo's, since `melt` walks one column at
a time. `var_name` names the column that holds the old column names, and `value_name` the column
that holds the readings. Sorted by time and station, with the columns in the original order, it is
the table Setup built, missing readings included:

In [6]:
back = long.sort_values(["time", "station"]).reset_index(drop=True)[["station", "time", "celsius"]]

print(back.equals(readings))

True


### A report of monthly means with pivot_table

Each station has hundreds of readings in each month, so the cell for Bergen in January has to
combine them. `pivot_table` takes the mean unless told otherwise:

In [7]:
by_month = readings.assign(month=readings["time"].dt.month)

by_month.pivot_table(index="station", columns="month", values="celsius").round(1)

month,1,2,3,4,5,6,7,8,9,10,11,12
station,,,,,,,,,,,,
Bergen,-0.9,0.1,3.2,7.5,12.1,15.5,16.9,15.9,12.7,8.3,3.8,0.4
Oslo,-2.4,-1.4,1.7,6.0,10.6,14.0,15.4,14.4,11.2,6.8,2.3,-1.1
Svalbard,-13.4,-12.4,-9.3,-5.0,-0.4,3.0,4.4,3.4,0.2,-4.2,-8.7,-12.1
Tromso,-5.4,-4.4,-1.3,3.0,7.6,11.0,12.4,11.4,8.2,3.8,-0.7,-4.1


`assign`, which the **Adding and Changing Columns** notebook covers, added a `month` column to a
copy of the readings, and the result is the same table the **Why Pandas** notebook made with
`groupby` and `unstack`.

### Counting frost hours with pivot_table

`aggfunc` names the function that combines each cell. A column of `True` and `False` sums to the
number of `True` values, so `aggfunc="sum"` on a frost flag counts the hours below freezing.
`margins=True` adds the totals as a row and a column named `All`:

In [8]:
frost = by_month.assign(frost=by_month["celsius"] < 0)

frost.pivot_table(index="station", columns="month", values="frost", aggfunc="sum", margins=True)

month,1,2,3,4,5,6,7,8,9,10,11,12,All
station,,,,,,,,,,,,,
Bergen,441,323,86,0,0,0,0,0,0,0,40,330,1220
Oslo,598,440,207,0,0,0,0,0,0,0,145,458,1848
Svalbard,744,672,720,706,408,72,0,39,341,705,720,744,5871
Tromso,744,668,493,98,0,0,0,0,0,54,418,728,3203
All,2527,2103,1506,804,408,72,0,39,341,759,1323,2260,12142


Svalbard was below freezing for every hour of January, 744 of them, and 5,871 hours over the year.
Svalbard's silent day counts as no frost here, because `NaN < 0` is `False`, as the **Missing Data**
notebook showed.

### unstack and stack on a two-level index

A groupby by two keys returns a Series whose index has two levels. Here it is a mean for every
station at each hour of the day:

In [9]:
hour = readings["time"].dt.hour.rename("hour")
by_hour = readings.groupby(["station", hour])["celsius"].mean().round(2)

print(by_hour.index.nlevels, "levels,", len(by_hour), "values")
by_hour.head(3)

2 levels, 96 values


station  hour
Bergen   0       5.88
         1       5.40
         2       5.10
Name: celsius, dtype: float64

`unstack` moves the inner level, the hour, into the columns. `unstack(level=0)` moves the outer
level, the station, instead, which gives a table with a row for each hour:

In [10]:
hour_by_station = by_hour.unstack(level=0)

print(hour_by_station.shape)
hour_by_station.head(4)

(24, 4)


station,Bergen,Oslo,Svalbard,Tromso
hour,,,,
0,5.88,4.38,-6.60,1.38
1,5.40,3.90,-7.08,0.90
2,5.10,3.60,-7.38,0.60
3,5.00,3.50,-7.48,0.50


Every station is coldest at 03:00 and warmest at 15:00, the daily swing of the formula, which
`idxmin` and `idxmax` find in each column:

In [11]:
print(hour_by_station.idxmin().to_dict())
print(hour_by_station.idxmax().to_dict())

{'Bergen': 3, 'Oslo': 3, 'Svalbard': 3, 'Tromso': 3}
{'Bergen': 15, 'Oslo': 15, 'Svalbard': 15, 'Tromso': 15}


`stack` goes the other way, from columns back into an inner level of the index, so the 24 rows of 4
columns become 96 values again:

In [12]:
print(hour_by_station.stack().shape)

(96,)


### Choosing between long and wide

The two shapes hold the same values, and each makes a different kind of question short.

| Question | Easier on |
|---|---|
| a mean, a count or a filter over every station at once | long, with `groupby` |
| a difference or a ratio between two stations at the same hour | wide, with column arithmetic |
| a table for a person to read | wide, with `pivot_table` |
| adding a station | long, which needs new rows and no new column |

Keep the data long, as it usually arrives from a database or a log, and make it wide for the
question or the report that needs it.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/14-reshaping-solutions.ipynb).

**1.** Using `wide`, count the hours in which Svalbard was more than 8 degrees colder than Tromso.

In [13]:
# your code here

**2.** Make a `pivot_table` of each station's highest reading in each month, with `aggfunc="max"`, and find the month in which every station reached its highest.

In [14]:
# your code here

**3.** Make a table of monthly means with `pivot_table`, then `melt` it back to a long table with the columns `station`, `month` and `celsius`, and print its shape.

In [15]:
# your code here

**4.** Make the frost table again without `margins`, and print the station with the most frost hours in March, and how many.

In [16]:
# your code here

**5.** Make a table with a row for each station and a column for each day of the week, holding the mean reading, with `groupby` on the station and on `readings["time"].dt.day_name()`, which gives each reading's day, and then `unstack`.

In [17]:
# your code here

## Common errors

### ValueError: Index contains duplicate entries, cannot reshape

In [18]:
by_month.pivot(index="station", columns="month", values="celsius")

ValueError: Index contains duplicate entries, cannot reshape

`pivot` puts one value in each cell, and Bergen in January has 744 readings. When a pair of row and
column labels repeats, `pivot` cannot choose, so it raises. Use `pivot_table`, which combines the
values in each cell, and name the function with `aggfunc` if a mean is not what you want.

### No error, and a fraction where a count was meant

In [19]:
frost.pivot_table(index="station", columns="month", values="frost").iloc[:, :3].round(3)

month,1,2,3
station,,,
Bergen,0.593,0.481,0.116
Oslo,0.804,0.655,0.278
Svalbard,1.000,1.000,0.968
Tromso,1.000,0.994,0.663


Without `aggfunc`, `pivot_table` took the mean of each cell, and the mean of `True` and `False` is
the share of hours that were frosty: 0.593 for Bergen in January, where the count was 441. The table
looks reasonable and is a different quantity. Write `aggfunc="sum"` for a count, and `aggfunc` on
every `pivot_table` whose function matters.

### No error, and 24 readings fewer after stack

In [20]:
print(len(readings), "readings")
print(len(wide.stack()), "values after stack")
print(len(wide.stack(future_stack=True)), "with future_stack=True")

35040 readings
35016 values after stack
35040 with future_stack=True


`stack` in pandas 2.2 drops missing values by default, so Svalbard's 24 silent hours vanished and
nothing said so. A count or a mean computed afterward still runs, on 35,016 values.
`future_stack=True` uses the new implementation, which the pandas documentation says replaces the
current one in pandas 3.0, and it keeps them, as `melt` does.

### KeyError: "The following id_vars or value_vars are not present in the DataFrame: ['Olso']"

In [21]:
wide.reset_index().melt(id_vars="time", value_vars=["Bergen", "Olso"])

KeyError: "The following id_vars or value_vars are not present in the DataFrame: ['Olso']"

`value_vars` names the columns to turn into rows, and one name is misspelled. The message names the
column it could not find. `wide.columns.tolist()` lists the names to copy from.

## Recap

- Make a long table wide with `pivot`, when no pair of row and column labels appears twice
- Combine many values per cell with `pivot_table`, and choose the function with `aggfunc`
- Make a wide table long with `melt`, naming the columns to keep in `id_vars`
- Move an index level into the columns with `unstack`, and back with `stack(future_stack=True)`
- Choose the shape from the question: long for groupby, wide for comparisons across stations

## What is next

The **Merging** notebook joins the readings to the stations' latitudes on the station's name, and
shows what the four kinds of join do with Kirkenes, which has no readings.

---

&#8592; **Previous:** [Aggregating](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/13-aggregating.ipynb)  &nbsp;·&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Merging](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/15-merging.ipynb) &#8594;
